In [1]:
from abc import ABC, abstractmethod
from typing import Dict, Type

In [2]:
class PaymentMethod(ABC):

    @abstractmethod
    def get_details(self) -> str:
        pass

    @abstractmethod
    def pay(self, amount: float) -> bool:
        pass

In [4]:
class RazorpayCardPayment(PaymentMethod):

    def __init__(self, card_number):
        self.card_number = card_number

    def get_details(self):
        return "Razorpay Card: " + self.card_number

    def pay(self, amount):
        print(f"Payment of ₹{amount} made using Razorpay Card")
        return True


class RazorpayUPIPayment(PaymentMethod):

    def __init__(self, upi_id):
        self.upi_id = upi_id

    def get_details(self):
        return "Razorpay UPI: " + self.upi_id

    def pay(self, amount):
        print(f"Payment of ₹{amount} made using Razorpay UPI")
        return True

In [5]:
class StripeCardPayment(PaymentMethod):

    def __init__(self, card_number):
        self.card_number = card_number

    def get_details(self):
        return "Stripe Card: " + self.card_number

    def pay(self, amount):
        print(f"Payment of ₹{amount} made using Stripe Card")
        return True


class StripeUPIPayment(PaymentMethod):

    def __init__(self, upi_id):
        self.upi_id = upi_id

    def get_details(self):
        return "Stripe UPI: " + self.upi_id

    def pay(self, amount):
        print(f"Payment of ₹{amount} made using Stripe UPI")
        return True

In [6]:
class FactoryPaymentMethod(ABC):

    factory: Dict[str, Type[PaymentMethod]] = {}

    @classmethod
    def get_payment_object(cls, method_type: str, **kwargs) -> PaymentMethod:
        if method_type not in cls.factory:
            raise ValueError("Invalid payment method")

        return cls.factory[method_type](**kwargs)

In [7]:
class RazorpayFactory(FactoryPaymentMethod):

    factory = {
        "card": RazorpayCardPayment,
        "upi": RazorpayUPIPayment
    }

In [8]:
class StripeFactory(FactoryPaymentMethod):

    factory = {
        "card": StripeCardPayment,
        "upi": StripeUPIPayment
    }

In [9]:
class Aggregator(ABC):

    def __init__(self, name):
        self.name = name

    @abstractmethod
    def call_get_payment_object(self, method_type: str, amount: float, **kwargs) -> bool:
        pass

In [10]:
class RazorpayAggregator(Aggregator):

    processing_fee = 2.0
    factory = RazorpayFactory

    def __init__(self):
        super().__init__("Razorpay")

    def call_get_payment_object(self, method_type: str, amount: float, **kwargs):
        payment_object = self.factory.get_payment_object(
            method_type,
            **kwargs
        )

        print(payment_object.get_details())
        print(f"Processing Fee: {self.processing_fee}%")

        return payment_object.pay(amount)

In [11]:
class StripeAggregator(Aggregator):

    processing_fee = 2.9
    factory = StripeFactory

    def __init__(self):
        super().__init__("Stripe")

    def call_get_payment_object(self, method_type: str, amount: float, **kwargs):
        payment_object = self.factory.get_payment_object(
            method_type,
            **kwargs
        )

        print(payment_object.get_details())
        print(f"Processing Fee: {self.processing_fee}%")

        return payment_object.pay(amount)

In [12]:
class AggregatorFactory:

    factory = {
        "stripe": StripeAggregator,
        "razorpay": RazorpayAggregator
    }

    @classmethod
    def get_aggregator_object(cls, aggregator_name: str) -> Aggregator:

        if aggregator_name not in cls.factory:
            raise ValueError("Invalid aggregator")

        return cls.factory[aggregator_name]()

In [13]:
try:
    aggregator_name = input("Select Aggregator (stripe/razorpay): ").lower()

    aggregator = AggregatorFactory.get_aggregator_object(
        aggregator_name
    )

    method_type = input("Select Method (card/upi): ").lower()

    if method_type == "card":
        card_number = input("Enter Card Number: ")

        amount = float(input("Enter Amount: "))

        result = aggregator.call_get_payment_object(
            method_type,
            amount,
            card_number=card_number
        )

    elif method_type == "upi":
        upi_id = input("Enter UPI ID: ")

        amount = float(input("Enter Amount: "))

        result = aggregator.call_get_payment_object(
            method_type,
            amount,
            upi_id=upi_id
        )

    else:
        raise ValueError("Invalid payment method")

    if result:
        print("Payment Successful")
    else:
        print("Payment Failed")

except ValueError as e:
    print("Error:", e)

Select Aggregator (stripe/razorpay):  razorpay
Select Method (card/upi):  card
Enter Card Number:  1234567890987654321
Enter Amount:  500


Razorpay Card: 1234567890987654321
Processing Fee: 2.0%
Payment of ₹500.0 made using Razorpay Card
Payment Successful
